# 08 - Desagregação de cargas e predição (módulos 12 e 13)

**Nota sobre o escopo do módulo 13**: este dataset não contém uma trajetória contínua de degradação com um evento de falha registrado — cada rótulo é uma sessão estacionária de ~4-6h com a condição já configurada desde o início (confirmado empiricamente na Fase 1 desta sessão). Por isso 'tempo restante até a falha' (RUL) não é implementado aqui; o que os dados suportam — classificação com probabilidade e previsão da assinatura futura — está demonstrado abaixo.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from harmonic_signatures.config import PROCESSED_DIR
from harmonic_signatures.disaggregation.dataset import build_disaggregation_dataset
from harmonic_signatures.disaggregation.model import evaluate_disaggregation
from harmonic_signatures.prediction.fault_classifier import predict_with_probability
from harmonic_signatures.deep_learning.windowing import make_sequence_windows
from harmonic_signatures.prediction.forecasting import FORECAST_ARCHITECTURES, train_forecaster

harm_matrix = pd.read_parquet(PROCESSED_DIR / 'combined_matrix.parquet')
feature_cols = [c for c in harm_matrix.columns if c.startswith('H') and c.endswith('_avg') and '_harm' not in c and '_inter' not in c][:20]

## Desagregação: máquina + falha de diodo/retificador a partir do sinal combinado

Validação leave-one-condição-combinada-out (só há 4 combinações no dataset — tratar como prova de conceito). O alvo 'máquina' é praticamente constante entre as 4 combinações (todas compartilham a mesma condição de máquina saudável), então seu R²=1.0 é um artefato dessa constância, não evidência de separação real — o alvo 'source' (o defeito) é o que de fato testa generalização aqui.

In [2]:
ds = build_disaggregation_dataset(harm_matrix, feature_cols)
report = evaluate_disaggregation(ds)
report

,held_out_condition,target,mae,r2
0,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 2),machine,0.000000,1.0
1,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 2),source,6.057278,0.3
2,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 3),machine,0.000000,1.0
3,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 3),source,1.505127,0.3
4,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 6),machine,0.000000,1.0
5,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 6),source,1.505962,0.3
6,MÁQUINA BALANCEADA E ALINHADA + RETIFICADOR CO...,machine,0.000000,1.0
7,MÁQUINA BALANCEADA E ALINHADA + RETIFICADOR CO...,source,6.814504,0.3


## Predição do tipo de falha com probabilidade

In [3]:
X = harm_matrix[feature_cols].fillna(0).to_numpy()
y = harm_matrix['label'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(X_train, y_train)
pred_df = predict_with_probability(clf, X_test[:10], list(clf.classes_))
pred_df[['predicted_label', 'probability']]

,predicted_label,probability
0,MÁQUINA BALANCEADA (S/ PARAF),0.606607
1,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 2),1.000000
2,"MÁQUINA DESBALANCEADA, DESALINHADA (RET 15 PARAF)",0.995000
3,DIODO 2,1.000000
4,DIODO 6,1.000000
5,MÁQUINA BALANCEADA E ALINHADA + (-DIODO 6),1.000000
6,RETIFICADOR COMPLETO,1.000000
7,MÁQUINA BALANCEADA E ALINHADO,0.810000
8,"MÁQUINA DESBALANCEADA, DESALINHADA (RET 15 PARAF)",1.000000
9,DIODO 5,1.000000


## Previsão da assinatura futura das harmônicas (forecasting sequencial)

In [4]:
wd = make_sequence_windows(harm_matrix, feature_cols, window_size=8, stride=4, forecast_horizon=1)
valid = ~np.isnan(wd.y_forecast).any(axis=1)
X_seq, y_seq = wd.X[valid], wd.y_forecast[valid]
idx = np.arange(len(X_seq))
train_idx, val_idx = train_test_split(idx, test_size=0.25, random_state=0, stratify=wd.groups[valid])

mean = X_seq[train_idx].reshape(-1, X_seq.shape[-1]).mean(axis=0)
std = X_seq[train_idx].reshape(-1, X_seq.shape[-1]).std(axis=0); std[std == 0] = 1.0
X_seq_n = (X_seq - mean) / std
y_mean = y_seq[train_idx].mean(axis=0); y_std = y_seq[train_idx].std(axis=0); y_std[y_std == 0] = 1.0
y_seq_n = (y_seq - y_mean) / y_std

from harmonic_signatures.prediction.forecasting import build_forecast_bilstm
result = train_forecaster(build_forecast_bilstm, {}, X_seq_n[train_idx], y_seq_n[train_idx], X_seq_n[val_idx], y_seq_n[val_idx], epochs=10, verbose=0)
print('val_loss (mse, normalizado):', result.val_loss, '| val_mae (normalizado):', result.val_mae)

val_loss (mse, normalizado): 0.04964058846235275 | val_mae (normalizado): 0.10793182253837585
